# Experimento de recall y ruido

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "reamember").exists():
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from omegaconf import OmegaConf
from sklearn.metrics.pairwise import cosine_similarity

from reamember.config import setGlobalSeed, setDeviceConfig
from reamember.datasets.text import TextDatasetWrapper
from reamember.eam.associative import NumpyAssociativeMemory as AssociativeMemory
from reamember.eam.mops import memorize
from reamember.pipes.text import (
    apply_text_noise,
    create_sonar_model,
    get_memory_batch_size,
    normalize_noise_level,
    text_reconstruction_metrics,
    get_scalar_config_value,
    get_experiment_path,
    load_embeddings_dataset,
    Quant,
 )

EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments"

In [3]:
DATASET_NAME = "npvinHnivqn/EnglishDictionary"
COLUMN = "definition"
SEED = 42

LATENT_DIM = 1024
MEMORY_DOMAIN = 16384
MEMORY_BATCH_SIZE = 64
FILLING_PERCENT = 1.0
SIGMA = 0.01
IOTA = 0.0
KAPPA = 0.0
XI = 0.0

TEXT_INDEX = 10
NOISE_LEVELS = np.linspace(0.0, 0.3, 11)
print(f"Noise levels: {NOISE_LEVELS*100}%")
CHARACTER_MASK = "_"

cfg = OmegaConf.create(
    {
        "app": {
            "dataset": DATASET_NAME,
            "column": COLUMN,
            "modality": "text",
            "seed": SEED,
            "noise": 0.0,
        },
        "neural": {
            "latent_dim": [LATENT_DIM],
        },
        "memory": {
            "domain": [MEMORY_DOMAIN],
            "batch_size": MEMORY_BATCH_SIZE,
            "filling": [FILLING_PERCENT],
            "sigma": SIGMA,
            "iota": IOTA,
            "kappa": KAPPA,
            "xi": XI,
        },
    }
 )

setGlobalSeed(SEED)
device = setDeviceConfig()

latent = int(get_scalar_config_value(cfg.neural.latent_dim))
domain = int(get_scalar_config_value(cfg.memory.domain))
batch_size = get_memory_batch_size(cfg, domain)
filling_percent = float(get_scalar_config_value(cfg.memory.filling))
sigma = float(get_scalar_config_value(cfg.memory.sigma))
iota = float(get_scalar_config_value(cfg.memory.iota))
kappa = float(get_scalar_config_value(cfg.memory.kappa))
xi = float(get_scalar_config_value(cfg.memory.xi))

Noise levels: [ 0.  3.  6.  9. 12. 15. 18. 21. 24. 27. 30.]%


In [ ]:
path = get_experiment_path(cfg, EXPERIMENTS_ROOT, latent)
embeddings_dataset = load_embeddings_dataset(path, device=device)
dataset = TextDatasetWrapper(
    dataset_name=cfg.app.dataset,
    column=cfg.app.column,
    seed=cfg.app.seed,
)

transformer = create_sonar_model(device)
all_embeddings = torch.cat(
    [embeddings_dataset.train.data, embeddings_dataset.test.data],
    dim=0,
)
quantizer = Quant(all_embeddings)

eam = AssociativeMemory(
    n=latent,
    m=domain,
    xi=xi,
    sigma=sigma,
    iota=iota,
    kappa=kappa,
)
eam = memorize(
    eam,
    dataset=all_embeddings,
    quantizer=quantizer,
    filling_percent=filling_percent,
    batch_size=batch_size,
)

original_text = str(dataset.test[TEXT_INDEX])

state = {
    "cfg": cfg,
    "latent": latent,
    "domain": domain,
    "batch_size": batch_size,
    "filling_percent": filling_percent,
    "sigma": sigma,
    "iota": iota,
    "kappa": kappa,
    "xi": xi,
    "path": path,
    "embeddings_dataset": embeddings_dataset,
    "dataset": dataset,
    "transformer": transformer,
    "all_embeddings": all_embeddings,
    "quantizer": quantizer,
    "eam": eam,
    "original_text": original_text,
}

print(state)

parameter load: ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   628/628 100% 0:00:00

Output()

[INFO] Memorizing 111601 features with shape (111601, 1024)...


100%|██████████| 1744/1744 [00:39<00:00, 43.65it/s]


{'cfg': {'app': {'dataset': 'npvinHnivqn/EnglishDictionary', 'column': 'definition', 'modality': 'text', 'seed': 42, 'noise': 0.0}, 'neural': {'latent_dim': [1024]}, 'memory': {'domain': [16384], 'batch_size': 64, 'filling': [1.0], 'sigma': 0.01, 'iota': 0.0, 'kappa': 0.0, 'xi': 0.0}}, 'latent': 1024, 'domain': 16384, 'batch_size': 64, 'filling_percent': 1.0, 'sigma': 0.01, 'iota': 0.0, 'kappa': 0.0, 'xi': 0.0, 'path': PosixPath('/Users/roicort/GitHub/rEAMember/experiments/npvinHnivqn-EnglishDictionary/definition_1024'), 'embeddings_dataset': <reamember.datasets.embedding.EmbeddingDatasetWrapper object at 0x141f2b560>, 'dataset': <reamember.datasets.text.TextDatasetWrapper object at 0x142657ce0>, 'transformer': SONAR(
  (encode_model): TextToEmbeddingModelPipeline(
    (model): SonarTextTransformerEncoderModel(
      (encoder_frontend): TransformerEmbeddingFrontend(
        no_scale=False
        (embed): StandardEmbedding(num_embeddings=256206, embed_dim=1024, pad_idx=1, init_fn=init_

In [ ]:

def recall_single_text(original_text, cue_text, transformer, quantizer, eam, device):
    with torch.no_grad():
        cue_embedding = transformer.encode([cue_text], device=device)
    cue_quantized = quantizer.quantize(cue_embedding.detach().cpu().numpy(), eam.m)

    recalled_embeddings, recognized, weights = eam.batch_recall(cue_quantized)
    recognized = bool(np.asarray(recognized)[0])
    weight = float(np.asarray(weights, dtype=float)[0])

    result = {
        "original": original_text,
        "cue": cue_text,
        "recognized": recognized,
        "weight": weight,
        "reconstructed": None,
        "cosine": np.nan,
        "noise-original-cosine": np.nan,
        "l2": np.nan,
        "edit_distance": np.nan,
    }

    if not recognized:
        return result

    recalled_embedding = quantizer.dequantize(
        np.asarray(recalled_embeddings, dtype=float),
        eam.m,
    )[0]
    recalled_embedding = torch.as_tensor(
        recalled_embedding,
        dtype=torch.float32,
    ).unsqueeze(0)

    samples, _ = text_reconstruction_metrics(
        model=transformer,
        device=device,
        original_texts=[original_text],
        cue_texts=[cue_text],
        embeddings=recalled_embedding,
        batch_size=1,
    )
    sample = samples[0]
    result.update(
        {
            "reconstructed": sample["reconstructed"],
            "cosine": sample["cosine"],
            "l2": sample["l2"],
            "edit_distance": sample["edit_distance"],
        }
    )
    return result


def run_noise_sweep(original_text, noise_levels):
    rows = []
    for noise_level in noise_levels:
        normalized_noise = normalize_noise_level(noise_level)
        cue_text = apply_text_noise(
            original_text,
            noise_level=normalized_noise,
        )
        recall = recall_single_text(
            original_text=original_text,
            cue_text=cue_text,
            transformer=transformer,
            quantizer=quantizer,
            eam=eam,
            device=device,
        )
        rows.append(
            {
                "noise_level": float(normalized_noise),
                "masked_characters": int(sum(a != b for a, b in zip(original_text, cue_text))),
                **recall,
            }
        )

    return pd.DataFrame(rows)


## Parametros base

In [6]:
results = run_noise_sweep(
    original_text=original_text,
    noise_levels=NOISE_LEVELS,
 )

display(
    results[[
        "noise_level",
        "masked_characters",
        "recognized",
        "weight",
        "cosine",
        "l2",
        "edit_distance",
        "cue",
        "reconstructed",
    ]]
 )

/Users/roicort/GitHub/rEAMember/.venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:85.)
  return _C._get_float32_matmul_precision()
Encoding reconstructed texts: 100%|██████████| 1/1 [00:04<00:00,  4.38s/it]


,noise_level,masked_characters,recognized,weight,cosine,l2,edit_distance,cue,reconstructed
0,0.00,0,True,28.323242,0.989023,0.021202,0.0,one who belongs to the militia,one who belongs to the militia
1,0.03,1,False,0.000000,NaN,NaN,NaN,one who belongs to zhe militia,NaN
2,0.06,2,False,0.000000,NaN,NaN,NaN,one who bezongs to the milmtia,NaN
3,0.09,3,False,0.000000,NaN,NaN,NaN,one whl belodgs to the miyitia,NaN
4,0.12,3,False,0.000000,NaN,NaN,NaN,rne who belings to che militia,NaN
5,0.15,4,False,0.000000,NaN,NaN,NaN,one nho jelongs do the mizitia,NaN
6,0.18,5,False,0.000000,NaN,NaN,NaN,mne who belongf to pee eilitia,NaN
7,0.21,6,False,0.000000,NaN,NaN,NaN,one wvo nejonds tz the militla,NaN
8,0.24,6,False,0.000000,NaN,NaN,NaN,mnc who behonys to thb militiq,NaN
9,0.27,7,False,0.000000,NaN,NaN,NaN,onp who bdlongl to tho micuria,NaN


## Cambiando parametros

In [7]:
SIGMA = 0.01
IOTA = 0.0
KAPPA = 0.0
XI = 100

cfg.memory.sigma = SIGMA
cfg.memory.iota = IOTA
cfg.memory.kappa = KAPPA
cfg.memory.xi = XI

sigma = float(SIGMA)
iota = float(IOTA)
kappa = float(KAPPA)
xi = float(XI)

eam.sigma = sigma
eam.iota = iota
eam.kappa = kappa
eam.xi = xi

print(
    {
        "dataset": DATASET_NAME,
        "column": COLUMN,
        "device": str(device),
        "text_index": TEXT_INDEX,
        "latent": latent,
        "domain": domain,
        "batch_size": batch_size,
        "sigma": eam.sigma,
        "iota": eam.iota,
        "kappa": eam.kappa,
        "xi": eam.xi,
    }
)

results = run_noise_sweep(
    original_text=original_text,
    noise_levels=NOISE_LEVELS,
 )

display(
    results[[
        "noise_level",
        "masked_characters",
        "recognized",
        "weight",
        "cosine",
        "l2",
        "edit_distance",
        "cue",
        "reconstructed",
    ]]
 )

{'dataset': 'npvinHnivqn/EnglishDictionary', 'column': 'definition', 'device': 'mps', 'text_index': 10, 'latent': 1024, 'domain': 16384, 'batch_size': 64, 'sigma': 0.01, 'iota': 0.0, 'kappa': 0.0, 'xi': 100.0}


Encoding reconstructed texts: 100%|██████████| 1/1 [00:00<00:00, 13.64it/s]


,noise_level,masked_characters,recognized,weight,cosine,l2,edit_distance,cue,reconstructed
0,0.00,0,True,28.330078,0.989478,0.020812,0,one who belongs to the militia,one who belongs to the militia
1,0.03,1,True,27.990234,0.989329,0.021298,1,one who belongs to the miljtia,one who belongs to the miljtia
2,0.06,2,True,23.677734,0.826406,0.121156,4,onu who belojgs to the militia,uno who beloojs to the militia
3,0.09,3,True,23.708984,0.945428,0.065201,9,one who bslongs to the mdlitil,one who bslongs to the mdlitil until
4,0.12,3,True,23.926758,0.891414,0.102626,4,one who helongs to whs militia,one who helongs to ws' militia
5,0.15,4,True,24.145508,0.950438,0.082443,4,one wzo bulohgs to the miqitia,One wzo bulohgs to the miqitia
6,0.18,5,True,21.774414,0.866654,0.117999,9,ofe wqo bzlqngn to the militia,whoo wqo zblqngn to the militia
7,0.21,6,True,23.040039,0.932549,0.074955,7,one pho delougs to thv misitza,one pho delougs to thtv misitsa
8,0.24,6,True,24.257812,0.980669,0.039418,7,one who vehongs to uhe rimitma,one who vehong to uhe rimitma
9,0.27,7,True,22.422852,0.947369,0.069117,6,one who pelotqs wo tho mixioia,one who pelotqs wo the mixioia


In [8]:
fig = px.line(
    results,
    x="noise_level",
    y="edit_distance",
    title=f"Edit Distance vs Noise Level sigma={SIGMA} xi={XI}",
    labels={"noise_level": "Noise Level", "edit_distance": "Edit Distance"},
)
fig.show()

fig = px.line(
    results,
    x="noise_level",
    y="cosine",
    title=f"Cosine Similarity vs Noise Level sigma={SIGMA} xi={XI}",
    labels={"noise_level": "Noise Level", "cosine": "Cosine Similarity"},
)
fig.show()

### Changing Sigma

In [25]:
SIGMA = 0.06
IOTA = 0.0
KAPPA = 0.0
XI = 100

cfg.memory.sigma = SIGMA
cfg.memory.iota = IOTA
cfg.memory.kappa = KAPPA
cfg.memory.xi = XI

sigma = float(SIGMA)
iota = float(IOTA)
kappa = float(KAPPA)
xi = float(XI)

eam.sigma = sigma
eam.iota = iota
eam.kappa = kappa
eam.xi = xi

print(
    {
        "dataset": DATASET_NAME,
        "column": COLUMN,
        "device": str(device),
        "text_index": TEXT_INDEX,
        "latent": latent,
        "domain": domain,
        "batch_size": batch_size,
        "sigma": eam.sigma,
        "iota": eam.iota,
        "kappa": eam.kappa,
        "xi": eam.xi,
    }
)

results = run_noise_sweep(
    original_text=original_text,
    noise_levels=NOISE_LEVELS,
 )

display(
    results[[
        "noise_level",
        "masked_characters",
        "recognized",
        "weight",
        "cosine",
        "l2",
        "edit_distance",
        "cue",
        "reconstructed",
    ]]
 )

{'dataset': 'npvinHnivqn/EnglishDictionary', 'column': 'definition', 'device': 'mps', 'text_index': 10, 'latent': 1024, 'domain': 16384, 'batch_size': 64, 'sigma': 0.06, 'iota': 0.0, 'kappa': 0.0, 'xi': 100.0}


Encoding reconstructed texts: 100%|██████████| 1/1 [00:00<00:00, 14.29it/s]


,noise_level,masked_characters,recognized,weight,cosine,l2,edit_distance,cue,reconstructed
0,0.00,0,True,28.219727,0.720411,0.108353,0,one who belongs to the militia,one who belongs to the militia
1,0.03,1,True,25.993164,0.750638,0.125122,4,one who belongs to whe militia,a one who belongs to whey militia
2,0.06,2,True,27.048828,0.733103,0.116957,3,bne who belongd to the militia,bne who belonged to the militia
3,0.09,3,True,25.629883,0.606762,0.170187,15,ont zho belongs to the mqlitia,angt zho ke is owned to the mallytia
4,0.12,3,True,25.602539,0.606228,0.169816,14,ony who beltngs to the qilitia,ony that bellts knongs to the quiliattia
5,0.15,4,True,25.743164,0.619010,0.156859,8,one who belonms to tho mificia,one who loins to the mitchie
6,0.18,5,True,25.007812,0.677662,0.157802,10,onl who belongo ti rhe wilitia,onlyon who Belongs to the rhe Wilitiae
7,0.21,6,True,26.362305,0.506150,0.176214,15,jnx who belyggs to the mioitya,lng who are belligerent to the miotia
8,0.24,6,True,24.746094,0.562225,0.202655,8,ove who belwngs io khe mirbtia,Ave who belongs to bheya mirti
9,0.27,7,True,24.273438,0.674827,0.163428,22,one who seldxgs ho ohe milrtka,an adult sldxgs who ho oh my milrtna


In [26]:
fig = px.line(
    results,
    x="noise_level",
    y="edit_distance",
    title=f"Edit Distance vs Noise Level sigma={SIGMA} xi={XI}",
    labels={"noise_level": "Noise Level", "edit_distance": "Edit Distance"},
)
fig.show()

fig = px.line(
    results,
    x="noise_level",
    y="cosine",
    title=f"Cosine Similarity vs Noise Level sigma={SIGMA} xi={XI}",
    labels={"noise_level": "Noise Level", "cosine": "Cosine Similarity"},
)
fig.show()